# AML Sentinel — Feature Engineering
### Building all 18 features across 3 groups
- **Group 1:** Transactional Features (9 features)
- **Group 2:** Account-Level Features (7 features)
- **Group 3:** Graph Features (2 features)

In [1]:
# ── Cell 1: Imports & Setup ────────────────────────────────────
import sys
import os

sys.path.append(r"C:\Users\VatsaL\Desktop\Datasets\AML_Sentinel")
from config import *

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, hour, dayofweek, log1p, when, round as spark_round,
    lit, count, countDistinct, sum, avg, stddev,
    max, min, desc
)
from pyspark.sql.window import Window

py_round = round  # save Python built-in before PySpark overwrites

spark = SparkSession.builder \
    .appName("AML_Feature_Engineering") \
    .config("spark.driver.memory", SPARK_DRIVER_MEMORY) \
    .config("spark.sql.shuffle.partitions", SPARK_SHUFFLE_PARTITIONS) \
    .config("spark.pyspark.python", PYSPARK_PYTHON) \
    .config("spark.pyspark.driver.python", PYSPARK_PYTHON) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark Session started!")
print(f"   Spark Version : {spark.version}")

✅ Spark Session started!
   Spark Version : 4.1.1


In [2]:
# ── Cell 2: Load Data ──────────────────────────────────────────
print("📥 Loading transaction parquet...")
trans_df = spark.read.parquet(TRANS_PARQUET)
trans_df = trans_df.withColumnRenamed("Account.1", "To Account")
total = trans_df.count()
print(f"✅ Loaded {total:,} transactions")
print(f"   Columns: {trans_df.columns}")

📥 Loading transaction parquet...
✅ Loaded 31,898,238 transactions
   Columns: ['Timestamp', 'From Bank', 'Account', 'To Bank', 'To Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


## Group 1 — Transactional Features (9 features)

In [3]:
# ── Cell 3: Group 1 — Transactional Features ───────────────────
print("⚙️  Building Group 1 — Transactional Features...")

# Feature 1: is_cross_border (10x higher laundering rate)
trans_df = trans_df.withColumn(
    "is_cross_border",
    when(col("From Bank") != col("To Bank"), 1).otherwise(0)
)
print("  ✅ Feature 1: is_cross_border")

# Feature 2: hour_of_day
trans_df = trans_df.withColumn("hour_of_day", hour(col("Timestamp")))
print("  ✅ Feature 2: hour_of_day")

# Feature 3: is_peak_hour (11am-1pm highest laundering rate)
trans_df = trans_df.withColumn(
    "is_peak_hour",
    when(col("hour_of_day").between(11, 13), 1).otherwise(0)
)
print("  ✅ Feature 3: is_peak_hour")

# Feature 4: day_of_week (1=Sunday ... 7=Saturday)
trans_df = trans_df.withColumn("day_of_week", dayofweek(col("Timestamp")))
print("  ✅ Feature 4: day_of_week")

# Feature 5: is_weekend (Sat/Sun = 2x higher rate)
trans_df = trans_df.withColumn(
    "is_weekend",
    when(col("day_of_week").isin(1, 7), 1).otherwise(0)
)
print("  ✅ Feature 5: is_weekend")

# Feature 6: payment_format_risk (ACH=3, Bitcoin=2, others=1, Wire/Reinvestment=0)
trans_df = trans_df.withColumn(
    "payment_format_risk",
    when(col("Payment Format") == "ACH",           3)
    .when(col("Payment Format") == "Bitcoin",      2)
    .when(col("Payment Format") == "Cash",         1)
    .when(col("Payment Format") == "Cheque",       1)
    .when(col("Payment Format") == "Credit Card",  1)
    .when(col("Payment Format") == "Wire",         0)
    .when(col("Payment Format") == "Reinvestment", 0)
    .otherwise(1)
)
print("  ✅ Feature 6: payment_format_risk")

# Feature 7: currency_risk_score (1-5 based on EDA laundering rates)
trans_df = trans_df.withColumn(
    "currency_risk_score",
    when(col("Payment Currency").isin("UK Pound", "Ruble"),                     5)
    .when(col("Payment Currency").isin("Euro", "Yen"),                          4)
    .when(col("Payment Currency").isin("US Dollar", "Yuan", "Rupee"),           3)
    .when(col("Payment Currency").isin("Australian Dollar", "Canadian Dollar"), 2)
    .when(col("Payment Currency").isin(
        "Shekel", "Brazil Real", "Mexican Peso",
        "Saudi Riyal", "Swiss Franc", "Bitcoin"),                               1)
    .otherwise(2)
)
print("  ✅ Feature 7: currency_risk_score")

# Feature 8: amount_log (log transform to handle skew)
trans_df = trans_df.withColumn(
    "amount_log",
    spark_round(log1p(col("Amount Paid")), 4)
)
print("  ✅ Feature 8: amount_log")

# Feature 9: is_near_threshold ($8K-$10K structuring window = 3x rate)
trans_df = trans_df.withColumn(
    "is_near_threshold",
    when((col("Amount Paid") >= 8000) & (col("Amount Paid") < 10000), 1).otherwise(0)
)
print("  ✅ Feature 9: is_near_threshold")

print("\n📊 Validation — Sample rows:")
trans_df.select([
    "is_cross_border", "hour_of_day", "is_peak_hour",
    "day_of_week", "is_weekend", "payment_format_risk",
    "currency_risk_score", "amount_log", "is_near_threshold",
    "Is Laundering"
]).show(5)

print("\n📊 Feature averages — Laundering vs Legitimate:")
for feat in ["is_cross_border", "is_weekend", "is_peak_hour",
             "payment_format_risk", "currency_risk_score", "is_near_threshold"]:
    laund = trans_df.filter(col("Is Laundering") == 1) \
                    .selectExpr(f"round(avg({feat})*100, 3) as pct").collect()[0][0]
    legit = trans_df.filter(col("Is Laundering") == 0) \
                    .selectExpr(f"round(avg({feat})*100, 3) as pct").collect()[0][0]
    print(f"  {feat:<25} Laundering: {str(laund):>8}%  |  Legitimate: {str(legit):>8}%")

print("\n✅ Group 1 Complete! 9 features built.")

⚙️  Building Group 1 — Transactional Features...
  ✅ Feature 1: is_cross_border
  ✅ Feature 2: hour_of_day
  ✅ Feature 3: is_peak_hour
  ✅ Feature 4: day_of_week
  ✅ Feature 5: is_weekend
  ✅ Feature 6: payment_format_risk
  ✅ Feature 7: currency_risk_score
  ✅ Feature 8: amount_log
  ✅ Feature 9: is_near_threshold

📊 Validation — Sample rows:
+---------------+-----------+------------+-----------+----------+-------------------+-------------------+----------+-----------------+-------------+
|is_cross_border|hour_of_day|is_peak_hour|day_of_week|is_weekend|payment_format_risk|currency_risk_score|amount_log|is_near_threshold|Is Laundering|
+---------------+-----------+------------+-----------+----------+-------------------+-------------------+----------+-----------------+-------------+
|              0|         11|           1|          5|         0|                  0|                  3|    1.4951|                0|            0|
|              0|         11|           1|          5|    

## Group 2 — Account-Level Features (7 features)

In [4]:
# ── Cell 4: Group 2 — Account-Level Features ───────────────────
print("⚙️  Building Group 2 — Account-Level Features...")

# Feature 10 & 12: fan_out_degree + tx_velocity
fan_out = trans_df.groupBy("Account").agg(
    countDistinct("To Account").alias("fan_out_degree"),
    count("*").alias("tx_velocity"),
    spark_round(sum("Amount Paid"), 2).alias("total_sent"),
    spark_round(avg("Amount Paid"), 2).alias("avg_sent")
)
trans_df = trans_df.join(fan_out, on="Account", how="left")
print("  ✅ Feature 10: fan_out_degree")
print("  ✅ Feature 12: tx_velocity")

# Feature 11: fan_in_degree
fan_in = trans_df.groupBy("To Account").agg(
    countDistinct("Account").alias("fan_in_degree"),
    spark_round(sum("Amount Paid"), 2).alias("total_received"),
    spark_round(avg("Amount Paid"), 2).alias("avg_received")
)
trans_df = trans_df.join(fan_in, on="To Account", how="left")
print("  ✅ Feature 11: fan_in_degree")

# Feature 13: amount_per_tx
trans_df = trans_df.withColumn(
    "amount_per_tx",
    spark_round(
        col("total_sent") / when(col("tx_velocity") > 0,
            col("tx_velocity")).otherwise(1), 2
    )
)
print("  ✅ Feature 13: amount_per_tx")

# Feature 14: amount_zscore_per_bank
bank_stats = trans_df.groupBy("From Bank").agg(
    spark_round(avg("Amount Paid"), 2).alias("bank_avg_amount"),
    spark_round(stddev("Amount Paid"), 2).alias("bank_std_amount")
)
trans_df = trans_df.join(bank_stats, on="From Bank", how="left")
trans_df = trans_df.withColumn(
    "amount_zscore_per_bank",
    spark_round(
        (col("Amount Paid") - col("bank_avg_amount")) /
        when(col("bank_std_amount") > 0,
             col("bank_std_amount")).otherwise(1), 4
    )
)
print("  ✅ Feature 14: amount_zscore_per_bank")

# Feature 15: bank_risk_score
bank_risk = trans_df.groupBy("From Bank").agg(
    count("*").alias("bank_total_tx"),
    sum("Is Laundering").alias("bank_laund_tx")
)
bank_risk = bank_risk.withColumn(
    "bank_risk_score",
    spark_round(
        col("bank_laund_tx") / when(col("bank_total_tx") > 0,
            col("bank_total_tx")).otherwise(1) * 100, 6
    )
)
trans_df = trans_df.join(
    bank_risk.select("From Bank", "bank_risk_score"),
    on="From Bank", how="left"
)
print("  ✅ Feature 15: bank_risk_score")

# Feature 16: is_high_fan_out
trans_df = trans_df.withColumn(
    "is_high_fan_out",
    when(col("fan_out_degree") > 100, 1).otherwise(0)
)
print("  ✅ Feature 16: is_high_fan_out")

# Drop intermediate columns
cols_to_drop = ["bank_avg_amount", "bank_std_amount",
                "bank_total_tx", "bank_laund_tx",
                "total_sent", "avg_sent",
                "total_received", "avg_received"]
for c in cols_to_drop:
    if c in trans_df.columns:
        trans_df = trans_df.drop(c)

print("\n📊 Validation — Account feature averages:")
for feat in ["fan_out_degree", "fan_in_degree", "tx_velocity",
             "amount_per_tx", "bank_risk_score", "is_high_fan_out"]:
    laund = trans_df.filter(col("Is Laundering") == 1) \
        .selectExpr(f"round(avg({feat}), 4) as v").collect()[0][0]
    legit = trans_df.filter(col("Is Laundering") == 0) \
        .selectExpr(f"round(avg({feat}), 4) as v").collect()[0][0]
    print(f"  {feat:<30} Laundering: {str(laund):>12}  |  Legitimate: {str(legit):>12}")

print("\n✅ Group 2 Complete! 7 features built.")

⚙️  Building Group 2 — Account-Level Features...
  ✅ Feature 10: fan_out_degree
  ✅ Feature 12: tx_velocity
  ✅ Feature 11: fan_in_degree
  ✅ Feature 13: amount_per_tx
  ✅ Feature 14: amount_zscore_per_bank
  ✅ Feature 15: bank_risk_score
  ✅ Feature 16: is_high_fan_out

📊 Validation — Account feature averages:
  fan_out_degree                 Laundering:    3650.3143  |  Legitimate:    2854.0005
  fan_in_degree                  Laundering:       6.9951  |  Legitimate:       4.6803
  tx_velocity                    Laundering:   70281.1011  |  Legitimate:   54982.3331
  amount_per_tx                  Laundering: 29170738.4306  |  Legitimate: 4390182.0464
  bank_risk_score                Laundering:        0.197  |  Legitimate:       0.1103
  is_high_fan_out                Laundering:       0.1189  |  Legitimate:       0.0928

✅ Group 2 Complete! 7 features built.


## Group 3 — Graph Features (2 features)

In [6]:
# ── Cell 5: Group 3 — Graph Features ──────────────────────────
print("⚙️  Building Group 3 — Graph Features...")

# Feature 17: is_in_cycle (STRONGEST FEATURE — 17.267% laundering rate)
print("  Building Feature 17: is_in_cycle (may take a few minutes)...")
forward_edges = trans_df.select(
    col("Account").alias("src"),
    col("To Account").alias("dst")
).distinct()

reverse_edges = trans_df.select(
    col("Account").alias("dst"),
    col("To Account").alias("src")
).distinct()

cycle_pairs = forward_edges.join(reverse_edges, on=["src", "dst"]) \
    .withColumn("cycle_flag", lit(1))

trans_df = trans_df.join(
    cycle_pairs.select(
        col("src").alias("Account"),
        col("dst").alias("To Account"),
        col("cycle_flag")
    ),
    on=["Account", "To Account"],
    how="left"
)
trans_df = trans_df.withColumn(
    "is_in_cycle",
    when(col("cycle_flag") == 1, 1).otherwise(0)
).drop("cycle_flag")
print("  ✅ Feature 17: is_in_cycle")

# Feature 18: is_hub_bank (top 20 banks by laundering count)
hub_banks_df = trans_df.filter(col("Is Laundering") == 1) \
    .groupBy("From Bank").count() \
    .orderBy(desc("count")).limit(20) \
    .toPandas()
hub_banks = hub_banks_df["From Bank"].tolist()

print(f"  Hub banks identified: {hub_banks}")
trans_df = trans_df.withColumn(
    "is_hub_bank",
    when(col("From Bank").isin(hub_banks), 1).otherwise(0)
)
print("  ✅ Feature 18: is_hub_bank")

print("\n✅ Group 3 Complete! 2 features built.")

⚙️  Building Group 3 — Graph Features...
  Building Feature 17: is_in_cycle (may take a few minutes)...
  ✅ Feature 17: is_in_cycle
  Hub banks identified: ['070', '020', '000', '011', '012', '006', '001', '02310', '0272142', '01226', '003', '00718', '01213', '027', '01208', '014', '00214', '00357', '025', '0293268']
  ✅ Feature 18: is_hub_bank

✅ Group 3 Complete! 2 features built.


In [8]:
# ── Cell 6: Final Validation & Save ───────────────────────────
all_features = [
    # Group 1
    "is_cross_border", "hour_of_day", "is_peak_hour",
    "day_of_week", "is_weekend", "payment_format_risk",
    "currency_risk_score", "amount_log", "is_near_threshold",
    # Group 2
    "fan_out_degree", "fan_in_degree", "tx_velocity",
    "amount_per_tx", "amount_zscore_per_bank",
    "bank_risk_score", "is_high_fan_out",
    # Group 3
    "is_in_cycle", "is_hub_bank"
]

print(f"Total features built: {len(all_features)}")
for i, f in enumerate(all_features, 1):
    print(f"  {i:>2}. {f}")

print("\n📊 Key Feature Validation:")
print(f"{'Feature':<30} {'Laundering Avg':>15} {'Legitimate Avg':>15} {'Ratio':>8}")
print("-" * 72)
key_features = [
    "is_in_cycle", "is_cross_border", "is_near_threshold",
    "is_weekend", "is_hub_bank", "is_high_fan_out"
]
for feat in key_features:
    laund = trans_df.filter(col("Is Laundering") == 1) \
        .selectExpr(f"round(avg({feat}), 4) as v").collect()[0][0]
    legit = trans_df.filter(col("Is Laundering") == 0) \
        .selectExpr(f"round(avg({feat}), 4) as v").collect()[0][0]
    ratio = py_round(float(laund) / (float(legit) if float(legit) > 0 else 0.0001), 2)
    print(f"  {feat:<28} {str(laund):>15} {str(legit):>15} {ratio:>7}x")

# Select final columns
model_cols = all_features + [
    "Account", "To Account", "From Bank", "To Bank",
    "Amount Paid", "Timestamp", "Is Laundering"
]
final_df = trans_df.select(model_cols)

# Fill nulls
null_fill = {
    "fan_in_degree": 0, "fan_out_degree": 0,
    "tx_velocity": 1, "amount_per_tx": 0,
    "amount_zscore_per_bank": 0, "bank_risk_score": 0,
    "is_high_fan_out": 0, "is_in_cycle": 0, "is_hub_bank": 0
}
final_df = final_df.fillna(null_fill)

print(f"\n💾 Saving features_final.parquet...")
final_df.write.mode("overwrite").parquet(FEATURES_FINAL_PARQUET)
print(f"✅ Saved to {FEATURES_FINAL_PARQUET}")
print(f"   Total rows     : {final_df.count():,}")
print(f"   Total columns  : {len(final_df.columns)}")
print(f"   Feature columns: {len(all_features)}")

spark.stop()
print("\n✅ Feature Engineering Complete! Ready for model training.")

Total features built: 18
   1. is_cross_border
   2. hour_of_day
   3. is_peak_hour
   4. day_of_week
   5. is_weekend
   6. payment_format_risk
   7. currency_risk_score
   8. amount_log
   9. is_near_threshold
  10. fan_out_degree
  11. fan_in_degree
  12. tx_velocity
  13. amount_per_tx
  14. amount_zscore_per_bank
  15. bank_risk_score
  16. is_high_fan_out
  17. is_in_cycle
  18. is_hub_bank

📊 Key Feature Validation:
Feature                         Laundering Avg  Legitimate Avg    Ratio
------------------------------------------------------------------------
  is_in_cycle                           0.1254          0.0829    1.51x
  is_cross_border                       0.9903           0.912    1.09x
  is_near_threshold                     0.0634          0.0221    2.87x
  is_weekend                            0.2614          0.1041    2.51x
  is_hub_bank                           0.2027          0.1435    1.41x
  is_high_fan_out                       0.1189          0.0928    1.